# 📱 Klasifikasi Spam SMS Menggunakan NLP & Machine Learning
### Metodologi: CRISP-DM (Cross-Industry Standard Process for Data Mining)

**Dataset:** SMS Spam Collection Dataset (UCI Machine Learning Repository)

| Info | Detail |
|------|--------|
| Tipe Masalah | Binary Text Classification |
| Teknik NLP | TF-IDF Vectorization |
| Model Utama | Multinomial Naive Bayes + SVM |
| Poin Plus | Eksplorasi Gradient Boosting + Word Cloud Analysis |

---

FASE 1 — Business Understanding

### Latar Belakang
Spam SMS merupakan ancaman nyata yang merugikan pengguna telepon seluler. Pesan spam dapat berupa penipuan (phishing), iklan tidak diinginkan, hingga tautan berbahaya. Berdasarkan laporan Statista (2023), lebih dari **45 miliar** pesan spam dikirim setiap harinya di seluruh dunia.

### Tujuan Bisnis
Membangun sistem klasifikasi otomatis yang mampu membedakan pesan SMS **spam** dari pesan **ham (bukan spam)** secara cepat dan akurat, untuk diintegrasikan ke aplikasi filter pesan.

### Tujuan Data Mining
Membangun model klasifikasi teks biner menggunakan teknik NLP yang mengklasifikasikan SMS sebagai:
- **HAM (0)** — pesan normal/sah
- **SPAM (1)** — pesan sampah/penipuan

Kriteria Keberhasilan
| Metrik | Target | Alasan |
|--------|--------|--------|
| Accuracy | ≥ 95% | Standar filter spam modern |
| Precision (Spam) | ≥ 90% | Hindari false positive (ham dianggap spam) |
| Recall (Spam) | ≥ 85% | Hindari false negative (spam lolos) |
| F1-Score | ≥ 88% | Keseimbangan precision & recall |

---
FASE 2 — Data Understanding

In [ ]:
# Install library yang dibutuhkan
!pip install -q wordcloud nltk scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
# ============================================================
# IMPORT SEMUA LIBRARY
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import re
import string
import warnings
import joblib
warnings.filterwarnings('ignore')

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud

# Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Model
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Evaluasi
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay, f1_score
)
from sklearn.pipeline import Pipeline

# Download NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('Semua library berhasil diimport!')
print(f'   Scikit-learn version: {__import__("sklearn").__version__}')

In [ ]:
# ============================================================
# LOAD DATASET — SMS Spam Collection
# Load langsung dari URL tanpa perlu akun Kaggle
# ============================================================
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'

df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

print(f'Dataset berhasil dimuat!')
print(f'   Shape: {df.shape}')
print(f'   Kolom: {df.columns.tolist()}')
df.head(10)

In [ ]:
# Informasi dasar dataset
print('=== INFORMASI DATASET ===')
df.info()
print()
print('=== DISTRIBUSI KELAS ===')
print(df['label'].value_counts())
print()
print(df['label'].value_counts(normalize=True).map('{:.2%}'.format))
print()
print('=== MISSING VALUES ===')
print(df.isnull().sum())
print()
print('=== DUPLIKASI ===')
print(f'Jumlah duplikat: {df.duplicated().sum()}')

In [ ]:
# Encode label: ham=0, spam=1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Tambah fitur statistik pesan
df['msg_length']    = df['message'].apply(len)
df['word_count']    = df['message'].apply(lambda x: len(x.split()))
df['char_count']    = df['message'].apply(lambda x: len(x.replace(' ', '')))
df['digit_count']   = df['message'].apply(lambda x: sum(c.isdigit() for c in x))
df['upper_count']   = df['message'].apply(lambda x: sum(c.isupper() for c in x))
df['punct_count']   = df['message'].apply(lambda x: sum(c in string.punctuation for c in x))

print('Fitur statistik berhasil ditambahkan!')
df[['label', 'message', 'msg_length', 'word_count', 'digit_count']].head()

In [ ]:
# ============================================================
# VISUALISASI EDA
# ============================================================
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig)

colors = {'ham': '#4CAF50', 'spam': '#F44336'}

# 1. Distribusi Kelas
ax1 = fig.add_subplot(gs[0, 0])
counts = df['label'].value_counts()
ax1.pie(counts, labels=['Ham', 'Spam'], autopct='%1.1f%%',
        colors=['#4CAF50', '#F44336'], startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax1.set_title('Distribusi Kelas', fontweight='bold')

# 2. Panjang Pesan
ax2 = fig.add_subplot(gs[0, 1])
for label in ['ham', 'spam']:
    df[df['label'] == label]['msg_length'].hist(
        ax=ax2, alpha=0.7, bins=30,
        color=colors[label], label=label.upper(), density=True
    )
ax2.set_title('Distribusi Panjang Pesan', fontweight='bold')
ax2.set_xlabel('Jumlah Karakter')
ax2.legend()

# 3. Jumlah Kata
ax3 = fig.add_subplot(gs[0, 2])
df.boxplot(column='word_count', by='label', ax=ax3,
           boxprops=dict(color='navy'),
           medianprops=dict(color='red', linewidth=2))
ax3.set_title('Jumlah Kata per Kelas', fontweight='bold')
ax3.set_xlabel('Label')
ax3.set_ylabel('Jumlah Kata')
plt.sca(ax3); plt.title('Jumlah Kata per Kelas')

# 4. Statistik perbandingan
ax4 = fig.add_subplot(gs[1, :])
stats = df.groupby('label')[['msg_length', 'word_count', 'digit_count', 'upper_count', 'punct_count']].mean()
stats.T.plot(kind='bar', ax=ax4, color=['#4CAF50', '#F44336'], alpha=0.8)
ax4.set_title('Rata-rata Statistik Pesan per Kelas', fontweight='bold')
ax4.set_xlabel('Fitur Statistik')
ax4.set_ylabel('Nilai Rata-rata')
ax4.legend(['Ham', 'Spam'])
ax4.tick_params(axis='x', rotation=0)

plt.suptitle('Exploratory Data Analysis — SMS Spam Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# WORD CLOUD — Kata paling sering muncul
# ============================================================
ham_text  = ' '.join(df[df['label'] == 'ham']['message'])
spam_text = ' '.join(df[df['label'] == 'spam']['message'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Word Cloud HAM
wc_ham = WordCloud(
    width=700, height=400,
    background_color='white',
    colormap='Greens',
    max_words=100
).generate(ham_text)

axes[0].imshow(wc_ham, interpolation='bilinear')
axes[0].set_title('Word Cloud — HAM (Pesan Normal)', fontsize=13, fontweight='bold', color='#2e7d32')
axes[0].axis('off')

# Word Cloud SPAM
wc_spam = WordCloud(
    width=700, height=400,
    background_color='white',
    colormap='Reds',
    max_words=100
).generate(spam_text)

axes[1].imshow(wc_spam, interpolation='bilinear')
axes[1].set_title('Word Cloud — SPAM', fontsize=13, fontweight='bold', color='#c62828')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Pesan SPAM sering mengandung kata: FREE, CALL, WIN, PRIZE, CLAIM, URGENT')

---
## 🔧 FASE 3 — Data Preparation

In [ ]:
# ============================================================
# 3.1 TEXT PREPROCESSING PIPELINE
# ============================================================
stemmer   = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Pipeline preprocessing teks:
    1. Lowercase
    2. Hapus URL, angka, tanda baca
    3. Tokenisasi
    4. Hapus stopwords
    5. Stemming
    """
    # 1. Lowercase
    text = text.lower()

    # 2. Hapus URL
    text = re.sub(r'http\S+|www\S+', '', text)

    # 3. Hapus angka
    text = re.sub(r'\d+', '', text)

    # 4. Hapus tanda baca
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 5. Hapus whitespace berlebih
    text = text.strip()

    # 6. Tokenisasi
    tokens = word_tokenize(text)

    # 7. Hapus stopwords & token pendek
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]

    # 8. Stemming
    tokens = [stemmer.stem(t) for t in tokens]

    return ' '.join(tokens)

# Terapkan ke seluruh dataset
df['message_clean'] = df['message'].apply(preprocess_text)

print('Preprocessing selesai!')
print()
# Contoh before vs after
for i in [0, 1, 5]:
    print(f'[{df["label"][i].upper()}]')
    print(f'  BEFORE: {df["message"][i][:80]}')
    print(f'  AFTER : {df["message_clean"][i][:80]}')
    print()

In [ ]:
# Top 15 kata setelah preprocessing
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, color, title in zip(
    axes,
    ['ham', 'spam'],
    ['#4CAF50', '#F44336'],
    ['Top 15 Kata — HAM', 'Top 15 Kata — SPAM']
):
    corpus = ' '.join(df[df['label'] == label]['message_clean'])
    word_freq = pd.Series(corpus.split()).value_counts().head(15)
    word_freq.sort_values().plot(kind='barh', ax=ax, color=color, alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frekuensi')

plt.tight_layout()
plt.savefig('top_words.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 3.2 Hapus duplikat & Split Data
# ============================================================
df_model = df.drop_duplicates(subset='message_clean').reset_index(drop=True)
print(f'Sebelum drop duplikat: {len(df)}')
print(f'Setelah drop duplikat: {len(df_model)}')

X = df_model['message_clean']
y = df_model['label_num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'\nData latih : {len(X_train)} sampel')
print(f'Data uji   : {len(X_test)} sampel')
print(f'Proporsi spam (train): {y_train.mean():.2%}')
print(f'Proporsi spam (test) : {y_test.mean():.2%}')

---
FASE 4 — Modeling

Menggunakan **TF-IDF Vectorizer** + pipeline klasifikasi.

Model yang dibandingkan:
1. **Multinomial Naive Bayes** — baseline NLP klasik
2. **Linear SVM** — powerful untuk teks
3. **Logistic Regression** — interpretable
4. **Random Forest** — ensemble
5. **Gradient Boosting** — eksplorasi mandiri (poin plus)

In [ ]:
# ============================================================
# 4.1 TF-IDF Vectorization
# ============================================================
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),   # unigram + bigram
    min_df=2,             # minimal muncul di 2 dokumen
    sublinear_tf=True     # log normalization
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'TF-IDF selesai!')
print(f'   Vocabulary size : {len(tfidf.vocabulary_):,} token')
print(f'   Train matrix    : {X_train_tfidf.shape}')
print(f'   Test matrix     : {X_test_tfidf.shape}')
print(f'   Matrix sparsity : {1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]):.2%}')

In [ ]:
# ============================================================
# 4.2 Perbandingan Model (Cross-Validation)
# ============================================================
models = {
    'Naive Bayes'        : MultinomialNB(alpha=0.1),
    'Linear SVM'         : LinearSVC(C=1.0, random_state=42, max_iter=2000),
    'Logistic Regression': LogisticRegression(C=1.0, random_state=42, max_iter=1000),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting'  : GradientBoostingClassifier(n_estimators=100, random_state=42),  # POIN PLUS
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
print('=== CROSS-VALIDATION (5-Fold) — Metrik: F1-Score (Spam) ===')
print(f'{"Model":<22} {"Mean F1":>9} {"Mean Acc":>10} {"Std":>8}')
print('-' * 55)

for name, model in models.items():
    f1_scores  = cross_val_score(model, X_train_tfidf, y_train, cv=cv, scoring='f1', n_jobs=-1)
    acc_scores = cross_val_score(model, X_train_tfidf, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results[name] = {'f1': f1_scores, 'acc': acc_scores}
    print(f'{name:<22} {f1_scores.mean():>9.4f} {acc_scores.mean():>10.4f} {f1_scores.std():>8.4f}')

In [ ]:
# Visualisasi perbandingan model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names  = list(cv_results.keys())
f1s    = [cv_results[n]['f1'].mean()  for n in names]
accs   = [cv_results[n]['acc'].mean() for n in names]
stds   = [cv_results[n]['f1'].std()   for n in names]
palette = ['#2196F3', '#F44336', '#9C27B0', '#4CAF50', '#FF9800']

# F1 Score
bars = axes[0].barh(names, f1s, xerr=stds, color=palette, alpha=0.85, capsize=5)
for bar, v in zip(bars, f1s):
    axes[0].text(v + 0.003, bar.get_y() + bar.get_height()/2, f'{v:.4f}', va='center', fontsize=9)
axes[0].set_xlabel('F1-Score (Spam)')
axes[0].set_title('Perbandingan F1-Score (5-Fold CV)', fontweight='bold')
axes[0].axvline(0.88, color='red', linestyle='--', alpha=0.5, label='Target 88%')
axes[0].set_xlim(0.75, 1.02)
axes[0].legend()

# Accuracy
bars2 = axes[1].barh(names, accs, color=palette, alpha=0.85)
for bar, v in zip(bars2, accs):
    axes[1].text(v + 0.001, bar.get_y() + bar.get_height()/2, f'{v:.4f}', va='center', fontsize=9)
axes[1].set_xlabel('Accuracy')
axes[1].set_title('Perbandingan Accuracy (5-Fold CV)', fontweight='bold')
axes[1].axvline(0.95, color='red', linestyle='--', alpha=0.5, label='Target 95%')
axes[1].set_xlim(0.88, 1.02)
axes[1].legend()

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 4.3 Train Model Terbaik: Linear SVM
# (Biasanya Linear SVM atau Naive Bayes unggul untuk teks)
# ============================================================
best_model = LinearSVC(C=1.0, random_state=42, max_iter=2000)
best_model.fit(X_train_tfidf, y_train)

print('Model Linear SVM berhasil dilatih!')
print(f'   Jumlah fitur : {X_train_tfidf.shape[1]:,}')
print(f'   Jumlah kelas : {len(best_model.classes_)}')

---
## FASE 5 — Evaluation

In [ ]:
# ============================================================
# 5.1 Evaluasi Lengkap pada Data Test
# ============================================================
y_pred = best_model.predict(X_test_tfidf)

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred)

print('=' * 55)
print('        HASIL EVALUASI MODEL TERBAIK (Linear SVM)')
print('=' * 55)
print(f'  Accuracy   : {acc:.4f} ({acc*100:.2f}%)')
print(f'  F1-Score   : {f1:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

In [ ]:
# Confusion Matrix + Visualisasi kesalahan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Ham', 'Spam'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix — Linear SVM', fontweight='bold')

# Perbandingan Semua Metrik Model
metric_data = {}
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    pred = model.predict(X_test_tfidf)
    metric_data[name] = {
        'Accuracy'  : accuracy_score(y_test, pred),
        'F1-Score'  : f1_score(y_test, pred),
    }

metric_df = pd.DataFrame(metric_data).T
metric_df.plot(kind='bar', ax=axes[1], color=['#2196F3', '#F44336'], alpha=0.85)
axes[1].set_title('Perbandingan Metrik Semua Model (Test Set)', fontweight='bold')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0.85, 1.01)
axes[1].tick_params(axis='x', rotation=20)
axes[1].legend()
axes[1].axhline(0.95, color='red', linestyle='--', alpha=0.4, label='Target Accuracy')

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5.2 Analisis Kata Paling Berpengaruh (Top Features)
# Poin Plus — Feature Analysis
# ============================================================
feature_names = tfidf.get_feature_names_out()
coef = best_model.coef_[0]

top_n = 20
top_spam_idx = np.argsort(coef)[-top_n:][::-1]
top_ham_idx  = np.argsort(coef)[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Top SPAM words
spam_words   = [feature_names[i] for i in top_spam_idx]
spam_weights = [coef[i]          for i in top_spam_idx]
axes[0].barh(spam_words[::-1], spam_weights[::-1], color='#F44336', alpha=0.85)
axes[0].set_title('Top 20 Kata Indikator SPAM', fontweight='bold', color='#c62828')
axes[0].set_xlabel('Bobot TF-IDF Coefficient')

# Top HAM words
ham_words   = [feature_names[i] for i in top_ham_idx]
ham_weights = [coef[i]          for i in top_ham_idx]
axes[1].barh(ham_words[::-1], [abs(w) for w in ham_weights[::-1]], color='#4CAF50', alpha=0.85)
axes[1].set_title('Top 20 Kata Indikator HAM', fontweight='bold', color='#2e7d32')
axes[1].set_xlabel('|Bobot TF-IDF Coefficient|')

plt.tight_layout()
plt.savefig('top_features.png', dpi=150, bbox_inches='tight')
plt.show()

print(' Kata kuat indikator SPAM:', spam_words[:10])
print(' Kata kuat indikator HAM :', ham_words[:10])

In [ ]:
# Analisis pesan yang salah diklasifikasi
test_df = pd.DataFrame({
    'message'    : X_test.values,
    'actual'     : y_test.values,
    'predicted'  : y_pred
})

false_pos = test_df[(test_df['actual'] == 0) & (test_df['predicted'] == 1)]  # Ham → Spam
false_neg = test_df[(test_df['actual'] == 1) & (test_df['predicted'] == 0)]  # Spam → Ham

print(f'False Positive (Ham dikira Spam): {len(false_pos)}')
print(f'False Negative (Spam lolos)     : {len(false_neg)}')
print()
print('Contoh False Positive (Ham yang salah diklasifikasi):')
for msg in false_pos['message'].head(3).values:
    print(f'  → {msg[:100]}')
print()
print('Contoh False Negative (Spam yang lolos):')
for msg in false_neg['message'].head(3).values:
    print(f'  → {msg[:100]}')

---
##  FASE 6 — Deployment

Simpan model dan vectorizer untuk digunakan di aplikasi Streamlit.

In [ ]:
# ============================================================
# 6.1 Simpan Model & Vectorizer
# ============================================================
# Re-train dengan SELURUH data untuk performa terbaik saat deploy
X_all = df_model['message_clean']
y_all = df_model['label_num']

tfidf_final = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)
X_all_tfidf = tfidf_final.fit_transform(X_all)

model_final = LinearSVC(C=1.0, random_state=42, max_iter=2000)
model_final.fit(X_all_tfidf, y_all)

joblib.dump(model_final,  'model_spam.pkl')
joblib.dump(tfidf_final,  'tfidf_vectorizer.pkl')

print('Model tersimpan      : model_spam.pkl')
print('Vectorizer tersimpan : tfidf_vectorizer.pkl')

In [ ]:
# ============================================================
# 6.2 Tes Prediksi Pesan Baru
# ============================================================
test_messages = [
    "WINNER!! You have been selected as a winner of £1000 prize. Call now to claim FREE!",
    "Hey, are you coming to the meeting tomorrow? Let me know.",
    "Congratulations! You've won a FREE iPhone. Click here: www.claim-prize.com",
    "Hi mom, I'll be home late tonight. Don't wait up for me.",
    "URGENT: Your account has been suspended. Verify now or lose access!",
]

print('=== TES PREDIKSI PESAN BARU ===')
print()
for msg in test_messages:
    msg_clean = preprocess_text(msg)
    msg_vec   = tfidf_final.transform([msg_clean])
    pred      = model_final.predict(msg_vec)[0]
    label     = '🚫 SPAM' if pred == 1 else '✅ HAM'
    print(f'{label}  |  {msg[:70]}')

In [ ]:
# Ringkasan Akhir
print('╔══════════════════════════════════════════════════╗')
print('║        RINGKASAN HASIL PROYEK ML — SPAM SMS      ║')
print('╠══════════════════════════════════════════════════╣')
print(f'║  Dataset      : SMS Spam Collection (5,572 SMS)  ║')
print(f'║  Teknik NLP   : TF-IDF (unigram + bigram)        ║')
print(f'║  Vocab Size   : 5,000 token                       ║')
print(f'║  Model Terbaik: Linear SVM                        ║')
print(f'║  Accuracy     : {acc*100:.2f}%                          ║')
print(f'║  F1-Score     : {f1:.4f}                          ║')
print('╠══════════════════════════════════════════════════╣')
print('║     Poin Plus: Gradient Boosting + Feature        ║')
print('║     Analysis + Word Cloud Visualization           ║')
print('╚══════════════════════════════════════════════════╝')